[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Memoria de corto y largo plazo.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 7: *Memoria y Deep Agents*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se distinguirá la memoria de un hilo (corto plazo) de la memoria que persiste entre conversaciones (largo plazo), y se construirá un agente que recuerda las preferencias de cada usuario.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del curso): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del curso (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue `TUTORIAL_SECRETOS.md`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este curso usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue TUTORIAL_SECRETOS.md: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.embeddings import init_embeddings
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

---
## 1. Tipos de memoria.

| Memoria | Alcance | Dónde vive | Ejemplo |
|---|---|---|---|
| **Corto plazo** | Un hilo (`thread_id`) | *Checkpointer* | "Como te dije hace dos mensajes…" |
| **Largo plazo semántica** | Entre hilos, por usuario | *Store* | "Ana prefiere Teams" |
| **Largo plazo episódica** | Entre hilos | *Store* | "La última vez el problema fue la VPN" |
| **Procedimental** | Todo el agente | *Prompt* o *store* | "En Finanzas, revisar primero el cierre contable" |

El *store* organiza la información en **espacios de nombres** (tuplas), por ejemplo `("preferencias", "ana.lopez")`, con pares clave-valor. Las etiquetas del espacio de nombres **no pueden contener puntos**: por eso `ana.lopez` se guarda como `ana_lopez`.

---
## 2. Herramientas con acceso al *store*.

`ToolRuntime` da a una herramienta acceso al *store* y al **contexto** de la ejecución (aquí, el usuario) sin que el modelo tenga que pasarlos como argumentos. Así el modelo no puede leer ni escribir la memoria de otro usuario.

* La siguiente celda definirá dos herramientas que leen y escriben la memoria del usuario actual.

In [ ]:
@dataclass
class Contexto:
    usuario: str


def espacio(usuario: str) -> tuple[str, str]:
    """Espacio de nombres de un usuario. Las etiquetas no pueden contener puntos."""
    return ("preferencias", usuario.replace(".", "_"))


@tool
def guardar_preferencia(clave: str, valor: str, runtime: ToolRuntime[Contexto]) -> str:
    """Guarda una preferencia duradera del usuario (canal de contacto, idioma, horario, estilo de respuesta)."""
    runtime.store.put(espacio(runtime.context.usuario), clave, {"valor": valor})
    return f"Guardado: {clave} = {valor}"


@tool
def leer_preferencias(runtime: ToolRuntime[Contexto]) -> dict:
    """Devuelve las preferencias guardadas del usuario actual."""
    return {i.key: i.value["valor"] for i in runtime.store.search(espacio(runtime.context.usuario))}

* La siguiente celda creará el agente con *checkpointer* (corto plazo) y *store* (largo plazo).

In [ ]:
store = InMemoryStore()
agente = create_agent(MODELO, tools=[guardar_preferencia, leer_preferencias],
    system_prompt=("Eres el asistente de soporte. Al iniciar, lee las preferencias del usuario y respétalas. "
                   "Guarda las preferencias nuevas que mencione."),
    context_schema=Contexto, checkpointer=InMemorySaver(), store=store)


def preguntar(texto: str, usuario: str, hilo: str) -> str:
    r = agente.invoke({"messages": [{"role": "user", "content": texto}]},
                      {"configurable": {"thread_id": hilo}}, context=Contexto(usuario=usuario))
    return r["messages"][-1].text

* La siguiente celda le dirá al agente una preferencia de `ana.lopez`.

In [ ]:
preguntar("Prefiero que me contacten por Teams y con respuestas cortas.", "ana.lopez", "hilo-1")

* La siguiente celda preguntará lo mismo desde un **hilo nuevo** de Ana y desde un hilo de Luis.

In [ ]:
{"ana, otro hilo": preguntar("¿Por dónde prefiero que me contacten?", "ana.lopez", "hilo-2"),
 "luis, otro hilo": preguntar("¿Por dónde prefiero que me contacten?", "luis.mora", "hilo-3")}

**Observa:** en un hilo nuevo no hay historial, pero Ana es reconocida por el *store*. Luis no ve las preferencias de Ana porque el espacio de nombres depende del contexto, no del modelo.

* La siguiente celda mostrará lo que quedó guardado.

In [ ]:
store.search(espacio("ana.lopez"))

**💡 Ideas clave de la sección 2:**
* El *checkpointer* recuerda un hilo; el *store* recuerda entre hilos.
* El contexto de ejecución (usuario) se pasa desde el código, no desde el modelo: es una frontera de seguridad.

### Ejercicio 1 · Olvidar.

Escribe la herramienta `olvidar_preferencias` que borre todo lo guardado del usuario actual (pista: `runtime.store.delete(espacio, clave)` para cada elemento). Agrégala al agente y prueba `"Olvida todo lo que sabes de mí"`.

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 2 · ¿Qué no guardar?

Lista tres tipos de información que un asistente de soporte **nunca** debería guardar en memoria de largo plazo y cómo lo impedirías técnicamente.

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Memoria semántica.

Crea un `InMemoryStore(index={"embed": init_embeddings(EMBEDDINGS), "dims": 3072})` (3072 es la dimensión de `gemini-embedding-001`; cámbiala si usas otro modelo de *embeddings*). Guarda cinco notas de un usuario y búscalas **por significado** con `store.search(espacio_de_notas, query="...")`.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## ✅ Cierre.

**Práctica:** terminar el ejercicio 1.

**Siguiente *notebook*:** `12_deep_agents.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>